# Shadow Tealbook A
### Purdue University — FOMC Research Project

**Part 1: Realized Data Pipeline — Interactive Charts**

All charts are interactive: hover for values, click legend to toggle series, drag to zoom, double-click to reset.

---

In [ ]:
import os, sys, warnings
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.subplots as sp
from datetime import datetime
from dateutil.relativedelta import relativedelta

warnings.filterwarnings('ignore')
os.environ['FRED_API_KEY'] = 'YOUR_FRED_KEY_HERE'
sys.path.insert(0, os.path.abspath('.'))

from data.fred_pulls import (
    get_series, pull_series, summary_snapshot,
    yield_curve_snapshot, output_gap, unemployment_gap,
    mortgage_spread, real_fed_funds
)
from config import FOMC_MEETINGS_2025, FOMC_MEETINGS_2026

BLUE, RED, GRAY, GREEN, ORANGE, NAVY2 = '#003087','#C41E3A','#808080','#2E7D32','#E65100','#1565C0'

# Rolling 7-year window — matches Fed Tealbook chart convention
START_DATE = (datetime.today() - relativedelta(years=7)).strftime('%Y-%m-01')

BASE_LAYOUT = dict(
    font=dict(family='Arial', size=11),
    plot_bgcolor='white', paper_bgcolor='white',
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0,
                bgcolor='rgba(0,0,0,0)', font=dict(size=10)),
    margin=dict(l=50, r=20, t=55, b=40),
    xaxis=dict(showgrid=True, gridcolor='#EEEEEE', zeroline=False,
               showline=True, linecolor='#CCCCCC', linewidth=1),
    yaxis=dict(showgrid=True, gridcolor='#EEEEEE', zeroline=False,
               showline=True, linecolor='#CCCCCC', linewidth=1),
)

# NBER recessions within the 7-year window (hardcoded, add as needed)
RECESSIONS = [('2020-02-01', '2020-04-01')]

def recession_shapes(xref='x'):
    """Gray shading for NBER recessions. xref can be 'x', 'x2', etc."""
    return [dict(type='rect', xref=xref, yref='paper',
                 x0=x0, x1=x1, y0=0, y1=1,
                 fillcolor='#CCCCCC', opacity=0.30,
                 line_width=0, layer='below')
            for x0, x1 in RECESSIONS]

def panel_titles(titles, rows, cols):
    """
    Return a list of annotation dicts that place panel titles
    inside the top-left of each subplot — Fed style, not Plotly default.
    titles: list of strings (left-to-right, top-to-bottom; use '' for empty)
    rows, cols: subplot grid dimensions
    """
    anns = []
    for idx, title in enumerate(titles):
        if not title:
            continue
        r = idx // cols + 1
        c = idx  % cols + 1
        xref = 'x' if (r == 1 and c == 1) else f'x{(r-1)*cols + c}'
        yref = 'y' if (r == 1 and c == 1) else f'y{(r-1)*cols + c}'
        anns.append(dict(
            text=f'<b>{title}</b>',
            xref=f'{xref} domain', yref=f'{yref} domain',
            x=0.0, y=1.08,
            xanchor='left', yanchor='bottom',
            font=dict(size=11, color='#222222', family='Arial'),
            showarrow=False,
        ))
    return anns

def layout(title, height=400, extra_shapes=None, extra_annotations=None):
    d = dict(**BASE_LAYOUT)
    shapes = recession_shapes()
    if extra_shapes:
        shapes += extra_shapes
    anns = extra_annotations or []
    d.update(
        title=dict(text=f'<b>{title}</b>',
                   font=dict(color=BLUE, size=14, family='Arial'), x=0),
        height=height,
        shapes=shapes,
        annotations=anns,
    )
    return d

print(f'Ready — {datetime.today().strftime("%B %d, %Y")}')
print(f'Chart window: {START_DATE} → present  (~7 years, rolling)')
print('Hover to inspect | Drag to zoom | Double-click to reset | Click legend to toggle')


---
## I. Summary Snapshot

In [ ]:
snap = summary_snapshot()
def color_status(v):
    return 'color: green' if isinstance(v, float) and v > 0 else ''
display(snap.style
    .set_table_styles([{'selector':'thead th','props':[('background-color','#003087'),('color','white'),('font-weight','bold'),('padding','6px 10px')]},
                        {'selector':'tbody tr:nth-child(even)','props':[('background-color','#f0f4fa')]},
                        {'selector':'td','props':[('font-size','11px'),('padding','4px 10px')]}])
    .set_caption(f'Key Indicators | {datetime.today().strftime("%B %d, %Y")}'))

---
## II-A. Output & Growth

In [ ]:
gdp = get_series('A191RL1Q225SBEA', start=START_DATE).dropna()
fig = go.Figure()
fig.add_trace(go.Bar(x=gdp.index, y=gdp.values, name='GDP Growth',
    marker_color=[BLUE if v >= 0 else RED for v in gdp.values]))
fig.add_hline(y=0, line_width=0.8, line_color='black')
fig.update_layout(**layout('Real GDP Growth — Annualized Quarterly Rate (%)'))
fig.update_yaxes(title='Percent, SAAR')
fig.show()

In [ ]:
gap = output_gap(START_DATE).dropna()
ip  = get_series('INDPRO', start=START_DATE).dropna()
tcu = pull_series('TCU',   start=START_DATE).dropna()

fig = sp.make_subplots(rows=1, cols=3, horizontal_spacing=0.08)

fig.add_trace(go.Scatter(x=gap.index, y=gap.values, name='Output Gap',
    fill='tozeroy', fillcolor='rgba(0,48,135,0.12)', line=dict(color=BLUE, width=1.8)), row=1, col=1)
fig.add_hline(y=0, line_width=0.7, line_color='black', row=1, col=1)

fig.add_trace(go.Scatter(x=ip.index, y=ip.values, name='IP YoY',
    fill='tozeroy', fillcolor='rgba(0,48,135,0.1)', line=dict(color=NAVY2, width=1.8)), row=1, col=2)
fig.add_hline(y=0, line_width=0.7, line_color='black', row=1, col=2)

fig.add_trace(go.Scatter(x=tcu.index, y=tcu.values, name='Cap Util',
    line=dict(color=GREEN, width=1.8)), row=1, col=3)
fig.add_hline(y=80, line_width=0.8, line_color=RED, line_dash='dot', row=1, col=3)

fig.update_layout(**layout('Output, Industrial Production & Capacity Utilization', height=420, extra_annotations=panel_titles(['Output Gap (% of potential, CBO)', 'Industrial Production (YoY %)', 'Capacity Utilization (%)'], 1, 3)))
fig.show()

---
## II-B. Consumption & Income

In [ ]:
pce   = get_series('PCECC96', start=START_DATE).dropna()
dpi   = get_series('DSPIC96', start=START_DATE).dropna()
sav   = pull_series('PSAVERT',start=START_DATE).dropna()
sent  = pull_series('UMCSENT',start=START_DATE).dropna()

fig = sp.make_subplots(rows=2, cols=2, vertical_spacing=0.12, horizontal_spacing=0.08)

fig.add_trace(go.Bar(x=pce.index, y=pce.values, name='PCE',
    marker_color=[BLUE if v >= 0 else RED for v in pce.values]), row=1, col=1)
fig.add_trace(go.Scatter(x=dpi.index, y=dpi.values, name='DPI',
    fill='tozeroy', fillcolor='rgba(46,125,50,0.1)', line=dict(color=GREEN, width=1.8)), row=1, col=2)
fig.add_hline(y=0, line_width=0.7, line_color='black', row=1, col=2)
fig.add_trace(go.Scatter(x=sav.index, y=sav.values, name='Saving Rate',
    fill='tozeroy', fillcolor='rgba(230,81,0,0.1)', line=dict(color=ORANGE, width=1.8)), row=2, col=1)
fig.add_trace(go.Scatter(x=sent.index, y=sent.values, name='Sentiment',
    line=dict(color=NAVY2, width=1.8)), row=2, col=2)

fig.update_layout(**layout('Consumption & Income', height=620, extra_annotations=panel_titles(['Real PCE Growth (ann. quarterly %)', 'Real Disp. Income Growth (YoY %)', 'Personal Saving Rate (%)', 'Michigan Consumer Sentiment'], 2, 2)))
fig.show()

---
## II-D. Housing

In [ ]:
starts   = pull_series('HOUST',       start=START_DATE).dropna()
permits  = pull_series('PERMIT',      start=START_DATE).dropna()
newsales = pull_series('HSN1F',       start=START_DATE).dropna()
hpi      = get_series('SPCS20RSA',    start=START_DATE).dropna()
mort     = pull_series('MORTGAGE30US',start=START_DATE).dropna()
mspd     = mortgage_spread(START_DATE).dropna()

fig = sp.make_subplots(rows=2, cols=3, vertical_spacing=0.14, horizontal_spacing=0.07)

fig.add_trace(go.Scatter(x=starts.index,  y=starts.values,  name='Starts',
    line=dict(color=BLUE, width=1.8)), row=1, col=1)
fig.add_trace(go.Scatter(x=permits.index, y=permits.values, name='Permits',
    line=dict(color=RED, width=1.3, dash='dash')), row=1, col=1)
fig.add_trace(go.Scatter(x=newsales.index, y=newsales.values, name='New Sales',
    line=dict(color=GREEN, width=1.8)), row=1, col=2)
fig.add_trace(go.Scatter(x=hpi.index, y=hpi.values, name='HPI YoY',
    fill='tozeroy', fillcolor='rgba(0,48,135,0.1)', line=dict(color=BLUE, width=1.8)), row=1, col=3)
fig.add_hline(y=0, line_width=0.7, line_color='black', row=1, col=3)
fig.add_trace(go.Scatter(x=mort.index, y=mort.values, name='Mortgage Rate',
    line=dict(color=ORANGE, width=1.8)), row=2, col=1)
fig.add_trace(go.Scatter(x=mspd.index, y=mspd.values, name='Mortgage Spread',
    fill='tozeroy', fillcolor='rgba(230,81,0,0.1)', line=dict(color=ORANGE, width=1.5)), row=2, col=2)

fig.update_layout(**layout('Housing Market', height=620, extra_annotations=panel_titles(['Housing Starts & Permits (thous., SAAR)', 'New Home Sales (thous.)', 'Case-Shiller HPI (YoY %)', '30-yr Mortgage Rate (%)', 'Mortgage Spread over 10-yr TSY (pp)', ''], 2, 3)))
fig.show()

---
## III. Labor Market

In [ ]:
# ── Labor Market (1): Payrolls ───────────────────────────────────────────────
# Change in Total Payroll Employment + ADP private comparison
# PAYEMS:       All employees, total nonfarm, monthly change (thous.)
# USPRIV:       All employees, total private, monthly change (thous.)
# ADPMNUSNERSA: ADP total nonfarm private, monthly change (thous.)

payrolls_lvl = pull_series('PAYEMS',       start=START_DATE).dropna()
private_lvl  = pull_series('USPRIV',       start=START_DATE).dropna()
adp_lvl      = pull_series('ADPMNUSNERSA', start=START_DATE).dropna()

payrolls = payrolls_lvl.diff().dropna()
private  = private_lvl.diff().dropna()
adp      = adp_lvl.diff().dropna()

fig = sp.make_subplots(rows=1, cols=2, horizontal_spacing=0.08)

fig.add_trace(go.Bar(x=payrolls.index, y=payrolls.values, name='Total Nonfarm (BLS)',
    marker_color=[BLUE if v >= 0 else RED for v in payrolls.values], opacity=0.8), row=1, col=1)
fig.add_hline(y=0, line_width=0.6, line_color='black', row=1, col=1)

fig.add_trace(go.Scatter(x=private.index,  y=private.values,  name='BLS Private',
    line=dict(color=BLUE, width=1.8)), row=1, col=2)
fig.add_trace(go.Scatter(x=adp.index, y=adp.values, name='ADP',
    line=dict(color=RED, width=1.5, dash='dash')), row=1, col=2)
fig.add_hline(y=0, line_width=0.6, line_color='black', row=1, col=2)

fig.update_layout(**layout('Payroll Employment', height=380,
    extra_annotations=panel_titles(
        ['Change in Total Nonfarm Payrolls (thous., SA)',
         'Change in Private Payrolls: BLS vs. ADP (thous., SA)'], 1, 2)))
fig.update_yaxes(title_text='Thousands', row=1, col=1)
fig.update_yaxes(title_text='Thousands', row=1, col=2)
fig.show()


In [ ]:
# ── Labor Market (2): Unemployment & Underutilization ────────────────────────
# UNRATE:      Unemployment rate (U-3), SA
# NROU:        Natural rate of unemployment (CBO), quarterly → interpolated monthly
# U5RATE:      U-5 (unemployed + discouraged + marginally attached), SA
# LNS12032194: Part-time for economic reasons, SA
# U6RATE:      U-6 (broadest measure), SA

unrate  = pull_series('UNRATE',      start=START_DATE).dropna()
nrou    = pull_series('NROU',        start=START_DATE).dropna().resample('MS').interpolate()
u5      = pull_series('U5RATE',      start=START_DATE).dropna()
u6      = pull_series('U6RATE',      start=START_DATE).dropna()
pteer   = pull_series('LNS12032194', start=START_DATE).dropna()

fig = sp.make_subplots(rows=1, cols=2, horizontal_spacing=0.08)

fig.add_trace(go.Scatter(x=unrate.index, y=unrate.values, name='U-3',
    line=dict(color=BLUE, width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=nrou.index, y=nrou.values, name='Natural Rate (CBO)',
    line=dict(color=RED, width=1.5, dash='dash')), row=1, col=1)

fig.add_trace(go.Scatter(x=u5.index, y=u5.values, name='U-5',
    line=dict(color=RED, width=1.8)), row=1, col=2)
fig.add_trace(go.Scatter(x=unrate.index, y=unrate.values, name='U-3',
    line=dict(color=GRAY, width=1.5)), row=1, col=2)
fig.add_trace(go.Scatter(x=pteer.index, y=pteer.values, name='Part-time econ. reasons',
    line=dict(color=BLUE, width=1.5, dash='dot')), row=1, col=2)

fig.update_layout(**layout('Unemployment', height=380,
    extra_annotations=panel_titles(
        ['Unemployment Rate vs. Natural Rate (%)',
         'Measures of Labor Underutilization (%)'], 1, 2)))
fig.update_yaxes(title_text='Percent', row=1, col=1)
fig.update_yaxes(title_text='Percent', row=1, col=2)
fig.show()


In [ ]:
# ── Labor Market (3): Participation & Employment-Population ──────────────────
# CIVPART:     Labor force participation rate (total), SA
# LNS12300060: Employment-population ratio, prime-age 25-54, SA
# EMRATIO:     Employment-population ratio (total), SA

lfpr   = pull_series('CIVPART',     start=START_DATE).dropna()
epop   = pull_series('EMRATIO',     start=START_DATE).dropna()
epop25 = pull_series('LNS12300060', start=START_DATE).dropna()

fig = sp.make_subplots(rows=1, cols=2, horizontal_spacing=0.08)

fig.add_trace(go.Scatter(x=lfpr.index, y=lfpr.values, name='LFPR (total)',
    line=dict(color=BLUE, width=1.8)), row=1, col=1)

fig.add_trace(go.Scatter(x=epop.index, y=epop.values, name='Total',
    line=dict(color=BLUE, width=1.8)), row=1, col=2)
fig.add_trace(go.Scatter(x=epop25.index, y=epop25.values, name='Prime-age (25-54)',
    line=dict(color=RED, width=1.5, dash='dash')), row=1, col=2)

fig.update_layout(**layout('Participation & Employment', height=380,
    extra_annotations=panel_titles(
        ['Labor Force Participation Rate (%)',
         'Employment-to-Population Ratio (%)'], 1, 2)))
fig.update_yaxes(title_text='Percent', row=1, col=1)
fig.update_yaxes(title_text='Percent', row=1, col=2)
fig.show()


In [ ]:
# ── Labor Market (4): JOLTS & Initial Claims ─────────────────────────────────
# JTS1000HIR:  Hires rate, total private, SA (% of employment)
# JTSJOR:      Job openings rate, total nonfarm, SA
# JTSQUR:      Quits rate, total nonfarm, SA
# IC4WSA:      Initial unemployment insurance claims, 4-week moving average (thous.)

hires  = pull_series('JTS1000HIR', start=START_DATE).dropna().rolling(3).mean()
openings = pull_series('JTSJOR',   start=START_DATE).dropna().rolling(3).mean()
quits  = pull_series('JTSQUR',     start=START_DATE).dropna().rolling(3).mean()
claims = pull_series('IC4WSA',     start=START_DATE).dropna()

fig = sp.make_subplots(rows=1, cols=2, horizontal_spacing=0.08)

fig.add_trace(go.Scatter(x=hires.index, y=hires.values, name='Hires',
    line=dict(color=BLUE, width=1.8)), row=1, col=1)
fig.add_trace(go.Scatter(x=openings.index, y=openings.values, name='Openings',
    line=dict(color=GREEN, width=1.8)), row=1, col=1)
fig.add_trace(go.Scatter(x=quits.index, y=quits.values, name='Quits',
    line=dict(color=RED, width=1.5, dash='dash')), row=1, col=1)

fig.add_trace(go.Scatter(x=claims.index, y=claims.values, name='Initial Claims (4-wk MA)',
    line=dict(color=ORANGE, width=1.8)), row=1, col=2)

fig.update_layout(**layout('JOLTS & Claims', height=380,
    extra_annotations=panel_titles(
        ['Hires, Openings & Quits Rates (%, 3-mo MA)',
         'Initial UI Claims, 4-week MA (thous.)'], 1, 2)))
fig.update_yaxes(title_text='Percent', row=1, col=1)
fig.update_yaxes(title_text='Thousands', row=1, col=2)
fig.show()


In [ ]:
# ── Labor Market (5): Unemployment & LFPR by Race/Ethnicity ─────────────────
# Unemployment rate by race (SA, monthly, 16+):
#   LNS14032183: Asian  |  LNS14000006: Black  |  LNS14000009: Hispanic  |  LNS14000003: White
# LFPR by race (all ages, substituting for 25-54 which is not available for all groups on FRED):
#   LNU01332183: Asian (NSA — STL seasonal adjustment applied)
#   LNS11300006: Black (SA)  |  LNS11300009: Hispanic (SA)  |  LNS11300003: White (SA)

from statsmodels.tsa.seasonal import STL

# Unemployment rates
ur_asian = pull_series('LNS14032183', start=START_DATE).dropna()
ur_black = pull_series('LNS14000006', start=START_DATE).dropna()
ur_hisp  = pull_series('LNS14000009', start=START_DATE).dropna()
ur_white = pull_series('LNS14000003', start=START_DATE).dropna()

# LFPR by race — Asian is NSA, apply STL
lfpr_asian_nsa = pull_series('LNU01332183', start=START_DATE).dropna()
stl = STL(lfpr_asian_nsa, period=12, robust=True).fit()
lfpr_asian = lfpr_asian_nsa - stl.seasonal

lfpr_black = pull_series('LNS11300006', start=START_DATE).dropna()
lfpr_hisp  = pull_series('LNS11300009', start=START_DATE).dropna()
lfpr_white = pull_series('LNS11300003', start=START_DATE).dropna()

fig = sp.make_subplots(rows=1, cols=2, horizontal_spacing=0.08)

PURPLE = '#6A0DAD'
for s, name, color, dash in [
    (ur_asian, 'Asian',    BLUE,   'solid'),
    (ur_black, 'Black',    RED,    'dash'),
    (ur_hisp,  'Hispanic', GRAY,   'dot'),
    (ur_white, 'White',    GREEN,  'solid'),
]:
    fig.add_trace(go.Scatter(x=s.index, y=s.values, name=name,
        line=dict(color=color, width=1.5, dash=dash)), row=1, col=1)

for s, name, color, dash in [
    (lfpr_asian, 'Asian (STL-SA)',  BLUE,  'solid'),
    (lfpr_black, 'Black',           RED,   'dash'),
    (lfpr_hisp,  'Hispanic',        GRAY,  'dot'),
    (lfpr_white, 'White',           GREEN, 'solid'),
]:
    fig.add_trace(go.Scatter(x=s.index, y=s.values, name=name,
        line=dict(color=color, width=1.5, dash=dash),
        showlegend=False), row=1, col=2)

fig.update_layout(**layout('Labor Market by Race/Ethnicity', height=400,
    extra_annotations=panel_titles(
        ['Unemployment Rate by Race/Ethnicity (%, SA, 3-mo MA)',
         'Labor Force Participation Rate by Race/Ethnicity (%, all ages)'], 1, 2)))
fig.update_yaxes(title_text='Percent', row=1, col=1)
fig.update_yaxes(title_text='Percent', row=1, col=2)
fig.show()


---
## IV. Prices & Inflation

In [ ]:
# ── Inflation (1): Headline CPI & PCE + Core PCE Measures ───────────────────
# CPIAUCSL:             CPI, all urban consumers, SA — YoY in code
# PCEPI:                PCE price index, SA — YoY in code
# PCEPILFE:             PCE ex. food & energy (standard core), SA — YoY in code
# PCETRIM12M159SFRBDAL: Trimmed mean PCE, 12-month rate, Dallas Fed, SA
# DPCXRG3M086SBEA:      Market-based PCE ex. food & energy, chain-type index — YoY in code

cpi      = pull_series('CPIAUCSL',             start=START_DATE).dropna().pct_change(12).mul(100)
pce      = pull_series('PCEPI',                start=START_DATE).dropna().pct_change(12).mul(100)
core_pce = pull_series('PCEPILFE',             start=START_DATE).dropna().pct_change(12).mul(100)
trim_pce = pull_series('PCETRIM12M159SFRBDAL', start=START_DATE).dropna()
mkt_pce  = pull_series('DPCXRG3M086SBEA',     start=START_DATE).dropna().pct_change(12).mul(100)

fig = sp.make_subplots(rows=1, cols=2, horizontal_spacing=0.08)

# Left: headline CPI vs PCE
for s, name, color, dash in [
    (cpi, 'CPI',  RED,  'solid'),
    (pce, 'PCE',  GRAY, 'solid'),
]:
    fig.add_trace(go.Scatter(x=s.index, y=s.values, name=name,
        line=dict(color=color, dash=dash, width=1.8)), row=1, col=1)
fig.add_hline(y=0, line_width=0.6, line_color='black', row=1, col=1)

# Right: core PCE measures
for s, name, color, dash in [
    (trim_pce, 'Trimmed mean PCE',             BLUE,  'solid'),
    (mkt_pce,  'Market-based PCE ex. F&E',     RED,   'solid'),
    (core_pce, 'PCE ex. food & energy',         GRAY,  'solid'),
]:
    fig.add_trace(go.Scatter(x=s.index, y=s.values, name=name,
        line=dict(color=color, dash=dash, width=1.8)), row=1, col=2)

fig.update_layout(**layout('Consumer Price Inflation', height=420,
    extra_annotations=panel_titles(
        ['Headline CPI & PCE (YoY %)',
         'Measures of Core PCE Inflation (YoY %)'], 1, 2)))
fig.update_yaxes(title_text='Percent', row=1, col=1)
fig.update_yaxes(title_text='Percent', row=1, col=2)
fig.show()


In [ ]:
# ── Inflation (2): Labor Cost Growth ─────────────────────────────────────────
# ECIWAG:        Employment cost index, wages & salaries, private industry, SA, quarterly
# CES0500000003: Average hourly earnings, total private nonfarm, SA — YoY in code
# HCOMPBS:       Business sector hourly compensation, all workers, SA, quarterly — YoY in code

eci  = pull_series('ECIWAG',       start=START_DATE).dropna().pct_change(4).mul(100)
ahe  = pull_series('CES0500000003',start=START_DATE).dropna().pct_change(12).mul(100)
comp = pull_series('HCOMPBS',      start=START_DATE).dropna().pct_change(4).mul(100)

fig = go.Figure()
for s, name, color, dash, width in [
    (eci,  'Employment cost index (private)',  BLUE,  'solid', 2.0),
    (ahe,  'Avg hourly earnings (priv. nonfarm)', RED,  'solid', 2.0),
    (comp, 'Compensation per hour (business)',    GRAY, 'solid', 1.6),
]:
    fig.add_trace(go.Scatter(x=s.index, y=s.values, name=name,
        line=dict(color=color, dash=dash, width=width)))
fig.add_hline(y=0, line_width=0.6, line_color='black')

fig.update_layout(**layout('Labor Cost Growth (YoY %)', height=400))
fig.update_yaxes(title_text='Percent')
fig.show()


In [ ]:
# ── Inflation (3): Oil Prices & Import Price Inflation ───────────────────────
# POILBREUSDM:  Global price of Brent crude, USD/barrel, monthly (IMF via FRED)
# DNRGRG3M086SBEA: PCE energy goods & services, chain-type price index — YoY in code
# IREXFDFLS:    Import price index ex. food & fuels (core import prices), NSA — YoY in code
# Note: CRB spot commodity index & tariff-adjusted core import prices are staff constructs;
#       not replicated here.

oil      = pull_series('POILBREUSDM',      start=START_DATE).dropna()
pce_nrg  = pull_series('DNRGRG3M086SBEA', start=START_DATE).dropna().pct_change(12).mul(100)
imp_core = pull_series('IREXFDFLS',       start=START_DATE).dropna().pct_change(12).mul(100)

fig = sp.make_subplots(rows=1, cols=2, horizontal_spacing=0.08,
    specs=[[{'secondary_y': False}, {'secondary_y': True}]])

# Left: Brent crude level
fig.add_trace(go.Scatter(x=oil.index, y=oil.values, name='Brent crude (USD/bbl)',
    fill='tozeroy', fillcolor='rgba(196,30,58,0.08)',
    line=dict(color=RED, width=1.8)), row=1, col=1)

# Right: PCE energy (right axis) + core import prices (left axis) — dual axis
fig.add_trace(go.Scatter(x=imp_core.index, y=imp_core.values,
    name='Core import prices (ex. food & fuels)',
    line=dict(color=GRAY, width=1.8)), row=1, col=2, secondary_y=False)
fig.add_trace(go.Scatter(x=pce_nrg.index, y=pce_nrg.values,
    name='PCE energy prices',
    line=dict(color=RED, width=1.8)), row=1, col=2, secondary_y=True)
fig.add_hline(y=0, line_width=0.6, line_color='black', row=1, col=2)

fig.update_layout(**layout('Commodity & Import Price Inflation', height=420,
    extra_annotations=panel_titles(
        ['Brent Crude Oil Price (USD/bbl)',
         'Energy & Import Price Inflation (YoY %)'], 1, 2)))
fig.update_yaxes(title_text='USD per barrel', row=1, col=1)
fig.update_yaxes(title_text='Core import prices (YoY %)', row=1, col=2, secondary_y=False)
fig.update_yaxes(title_text='PCE energy (YoY %)', row=1, col=2, secondary_y=True)
fig.show()


In [ ]:
# ── Inflation (4): Long-Term Inflation Expectations ───────────────────────────
# T5YIFRM:  5-year, 5-year forward inflation expectation rate, monthly, St. Louis Fed
#           Proxy for Tealbook "5-to-10-year TIPS compensation"
# MICH:     U. of Michigan: inflation expectation, next 12 months, median, monthly
#           Proxy for Tealbook "Michigan median next 5 to 10 years" (5-10yr not on FRED)
# EXPINF5YR: Cleveland Fed 5-year expected inflation (model-based)
#           Proxy for Tealbook "SPF PCE median next 10 years"
# Note: Michigan 5-to-10yr and SPF 10-yr PCE are not available as FRED time series.
#       Proxies are clearly labeled. Forecasts to be integrated at a later stage.

tips_fwd = pull_series('T5YIFRM',  start=START_DATE).dropna()
mich_1yr = pull_series('MICH',     start=START_DATE).dropna()
clev_5yr = pull_series('EXPINF5YR',start=START_DATE).dropna()

fig = go.Figure()
for s, name, color, dash in [
    (tips_fwd, '5yr5yr TIPS forward (proxy: TIPS compensation)',  RED,   'solid'),
    (mich_1yr, 'Michigan 1-yr ahead (proxy: Mich. 5-10yr)',       GRAY,  'solid'),
    (clev_5yr, 'Cleveland 5-yr expected (proxy: SPF 10-yr PCE)',  BLUE,  'dash'),
]:
    fig.add_trace(go.Scatter(x=s.index, y=s.values, name=name,
        line=dict(color=color, dash=dash, width=1.8)))
fig.add_hline(y=2.0, line_width=0.8, line_color='black', line_dash='dot',
    annotation_text='2%', annotation_position='top right')

fig.update_layout(**layout('Long-Term Inflation Expectations (%)', height=400))
fig.update_yaxes(title_text='Percent')
fig.show()


---
## V. Financial Conditions

In [ ]:
# Yield curve snapshot — current vs 1yr ago
yc    = yield_curve_snapshot()
one_yr = (pd.Timestamp.today() - pd.DateOffset(years=1)).strftime('%Y-%m-%d')
yc_1y = yield_curve_snapshot(as_of=one_yr)
order = [t for t in ['3M','6M','1Y','2Y','3Y','5Y','7Y','10Y','20Y','30Y'] if t in yc.index]
yc    = yc.reindex(order).dropna()
yc_1y = yc_1y.reindex(order).dropna()

fig = go.Figure()
fig.add_trace(go.Scatter(x=yc.index, y=yc['yield'].values,
    name=f'Current', mode='lines+markers',
    line=dict(color=BLUE, width=2.5), marker=dict(size=7)))
if not yc_1y.empty:
    fig.add_trace(go.Scatter(x=yc_1y.index, y=yc_1y['yield'].values,
        name='1 Year Ago', mode='lines+markers',
        line=dict(color=RED, width=1.5, dash='dash'), marker=dict(size=5)))
fig.update_layout(**{**BASE_LAYOUT, 'height':380, 'shapes':[],
    'title':dict(text='U.S. Treasury Yield Curve', font=dict(color=BLUE, size=13), x=0)})
fig.update_xaxes(title='Tenor')
fig.update_yaxes(title='Yield (%)')
fig.show()

In [ ]:
dgs2  = pull_series('DGS2',         start=START_DATE).dropna().resample('MS').mean()
dgs10 = pull_series('DGS10',        start=START_DATE).dropna().resample('MS').mean()
dgs30 = pull_series('DGS30',        start=START_DATE).dropna().resample('MS').mean()
spd   = pull_series('T10Y2Y',       start=START_DATE).dropna().resample('MS').mean()
spd3m = pull_series('T10Y3M',       start=START_DATE).dropna().resample('MS').mean()
ig    = pull_series('BAMLC0A0CM',   start=START_DATE).dropna().resample('MS').mean()
hy    = pull_series('BAMLH0A0HYM2', start=START_DATE).dropna().resample('MS').mean()
sp500 = pull_series('SP500',        start=START_DATE).dropna().resample('MS').mean()
vix   = pull_series('VIXCLS',       start=START_DATE).dropna().resample('MS').mean()
nfci  = pull_series('NFCI',         start=START_DATE).dropna()
sofr  = pull_series('SOFR',         start='2018-01-01').dropna().resample('MS').mean()

fig = sp.make_subplots(rows=3, cols=2, vertical_spacing=0.10, horizontal_spacing=0.08)

for s, name, color, dash in [(dgs2,'2-yr',RED,'dash'),(dgs10,'10-yr',BLUE,'solid'),(dgs30,'30-yr',GRAY,'dot')]:
    fig.add_trace(go.Scatter(x=s.index, y=s.values, name=name, line=dict(color=color, dash=dash, width=1.5)), row=1, col=1)

fig.add_trace(go.Scatter(x=spd.index, y=spd.values, name='10y-2y',
    fill='tozeroy', fillcolor='rgba(0,48,135,0.12)', line=dict(color=BLUE, width=1.8)), row=1, col=2)
fig.add_trace(go.Scatter(x=spd3m.index, y=spd3m.values, name='10y-3m',
    line=dict(color=RED, width=1.3, dash='dot')), row=1, col=2)
fig.add_hline(y=0, line_width=0.8, line_color='black', row=1, col=2)

fig.add_trace(go.Scatter(x=ig.index, y=ig.values, name='IG Spread',
    fill='tozeroy', fillcolor='rgba(0,48,135,0.1)', line=dict(color=BLUE, width=1.8)), row=2, col=1)
fig.add_trace(go.Scatter(x=hy.index, y=hy.values, name='HY Spread',
    fill='tozeroy', fillcolor='rgba(196,30,58,0.1)', line=dict(color=RED, width=1.8)), row=2, col=2)
fig.add_trace(go.Scatter(x=sp500.index, y=sp500.values, name='S&P 500',
    line=dict(color=BLUE, width=1.8)), row=3, col=1)
fig.add_trace(go.Scatter(x=nfci.index, y=nfci.values, name='NFCI',
    line=dict(color=NAVY2, width=1.5)), row=3, col=2)
fig.add_trace(go.Scatter(x=vix.index, y=vix.values, name='VIX',
    line=dict(color=ORANGE, width=1.2, dash='dot')), row=3, col=2)
fig.add_hline(y=0, line_width=0.6, line_color=NAVY2, line_dash='dot', row=3, col=2)

fig.update_layout(**layout('Financial Conditions', height=900, extra_annotations=panel_titles(['Treasury Yields (%)', '10yr-2yr & 10yr-3m Spreads (pp)', 'IG Corporate Spread (OAS, bps)', 'HY Corporate Spread (OAS, bps)', 'S&P 500 Index', 'VIX & Chicago Fed NFCI'], 3, 2)))
fig.show()

---
## VI. Monetary Policy Stance

In [ ]:
ffr   = pull_series('FEDFUNDS',  start=START_DATE).dropna().resample('MS').mean()
ub    = pull_series('DFEDTARU',  start='2015-01-01').dropna().resample('MS').mean()
rffr  = real_fed_funds(START_DATE).dropna()
m2    = get_series('M2SL',       start=START_DATE).dropna()
walcl = pull_series('WALCL',     start=START_DATE).dropna().resample('MS').mean() / 1e6
sloos = pull_series('DRTSCILM',  start=START_DATE).dropna()
ci    = pull_series('BUSLOANS',  start=START_DATE).dropna().resample('MS').mean()
ci_g  = ci.pct_change(12) * 100

fig = sp.make_subplots(rows=2, cols=3, subplot_titles=(
    'Federal Funds Rate (%)', 'Real Federal Funds Rate (%)',
    'M2 Growth (YoY %)', 'Fed Balance Sheet ($Tril)',
    'SLOOS: Net % Tightening C&I Standards', 'C&I Loan Growth (YoY %)'))

fig.add_trace(go.Scatter(x=ffr.index, y=ffr.values, name='Effective FFR',
    line=dict(color=BLUE, width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=ub.index, y=ub.values, name='Target Upper',
    line=dict(color=RED, width=1.3, dash='dash')), row=1, col=1)
fig.add_trace(go.Scatter(x=rffr.index, y=rffr.values, name='Real FFR',
    fill='tozeroy', fillcolor='rgba(0,48,135,0.12)', line=dict(color=BLUE, width=1.8)), row=1, col=2)
fig.add_hline(y=0, line_width=0.8, line_color='black', row=1, col=2)
fig.add_trace(go.Scatter(x=m2.index, y=m2.values, name='M2 YoY',
    fill='tozeroy', fillcolor='rgba(46,125,50,0.1)', line=dict(color=GREEN, width=1.8)), row=1, col=3)
fig.add_hline(y=0, line_width=0.6, line_color='black', row=1, col=3)
fig.add_trace(go.Scatter(x=walcl.index, y=walcl.values, name='Fed Assets',
    fill='tozeroy', fillcolor='rgba(0,48,135,0.1)', line=dict(color=NAVY2, width=1.8)), row=2, col=1)
fig.add_trace(go.Bar(x=sloos.index, y=sloos.values, name='SLOOS',
    marker_color=[RED if v > 0 else GREEN for v in sloos.values], opacity=0.8), row=2, col=2)
fig.add_hline(y=0, line_width=0.8, line_color='black', row=2, col=2)
fig.add_trace(go.Scatter(x=ci_g.dropna().index, y=ci_g.dropna().values, name='C&I Loan Growth',
    fill='tozeroy', fillcolor='rgba(230,81,0,0.1)', line=dict(color=ORANGE, width=1.8)), row=2, col=3)
fig.add_hline(y=0, line_width=0.6, line_color='black', row=2, col=3)

fig.update_layout(**layout('Monetary Policy Stance & Money/Banking', height=660))
fig.show()

---
## VII. International

In [ ]:
dollar = pull_series('RTWEXBGS',           start=START_DATE).dropna()
tb     = pull_series('BOPGSTB',            start=START_DATE).dropna()
ea_gdp = get_series('CLVMNACSCAB1GQEA19',  start=START_DATE).dropna()
eurusd = pull_series('DEXUSEU',            start=START_DATE).dropna().resample('MS').mean()
cnyusd = pull_series('DEXCHUS',            start=START_DATE).dropna().resample('MS').mean()
em_spd = pull_series('BAMLEMCBPIOAS',      start=START_DATE).dropna().resample('MS').mean()

fig = sp.make_subplots(rows=2, cols=3, subplot_titles=(
    'Real Broad Dollar Index (Jan 2006=100)', 'Trade Balance (goods & svcs, mil.$)',
    'Eurozone GDP Growth (annualized %)', 'EUR/USD',
    'CNY/USD', 'EM Corporate Bond Spread (bps)'))

fig.add_trace(go.Scatter(x=dollar.index, y=dollar.values, name='Real Broad $',
    line=dict(color=BLUE, width=1.8)), row=1, col=1)
fig.add_trace(go.Bar(x=tb.index, y=tb.values, name='Trade Balance',
    marker_color=RED, opacity=0.7), row=1, col=2)
fig.add_trace(go.Bar(x=ea_gdp.index, y=ea_gdp.values, name='EA GDP',
    marker_color=[BLUE if v >= 0 else RED for v in ea_gdp.values], opacity=0.8), row=1, col=3)
fig.add_hline(y=0, line_width=0.7, line_color='black', row=1, col=3)
fig.add_trace(go.Scatter(x=eurusd.index, y=eurusd.values, name='EUR/USD',
    line=dict(color=GREEN, width=1.8)), row=2, col=1)
fig.add_trace(go.Scatter(x=cnyusd.index, y=cnyusd.values, name='CNY/USD',
    line=dict(color=ORANGE, width=1.8)), row=2, col=2)
fig.add_trace(go.Scatter(x=em_spd.index, y=em_spd.values, name='EM Spread',
    fill='tozeroy', fillcolor='rgba(196,30,58,0.1)', line=dict(color=RED, width=1.8)), row=2, col=3)

fig.update_layout(**layout('International Economy & Trade', height=660))
fig.show()

---
## Diagnostic

In [ ]:
from data.fred_pulls import test_all_pulls
results = test_all_pulls(verbose=False)
failed  = results[results['Status'] == 'FAILED']
ok      = results[results['Status'] == 'OK']
print(f'✓ OK: {len(ok)} series  |  ✗ Failed: {len(failed)} series')
if len(failed) > 0:
    print('\nFailed:')
    display(failed[['Series ID','Label','Section']])

def color_status(v):
    if v == 'OK':     return 'color: green; font-weight: bold'
    if v == 'FAILED': return 'color: red;   font-weight: bold'
    return ''

display(results.style
    .map(color_status, subset=['Status'])
    .set_caption('Data Pull Diagnostic — Shadow Tealbook A'))